# 03 · Preparación del target y variables


Importo las librerías que voy a utilizar


In [1]:
from pathlib import Path
import pandas as pd

Cargo la base consolidada y defino la carpeta de salida


In [2]:
STAGING_FILE = Path("../data/staging/storm_events_details_2010_2025.parquet")

PROCESSED_DIR = Path("../data/processed")

PROCESSED_DIR.mkdir(parents=True,exist_ok=True)

df_storm = pd.read_parquet(STAGING_FILE)

print("Filas:", f"{df_storm.shape[0]:,}")
print("Columnas:", df_storm.shape[1])

Filas: 1,037,691
Columnas: 53


Reviso las primeras filas de la base


In [3]:
df_storm.head()

,begin_yearmonth,begin_day,begin_time,end_yearmonth,end_day,end_time,episode_id,event_id,state,state_fips,...,end_location,begin_lat,begin_lon,end_lat,end_lon,episode_narrative,event_narrative,data_source,source_year,source_file
0,201011,22,1531,201011,22,1532,46247,268262,ILLINOIS,17,...,BIG FOOT,42.4932,-88.5704,42.4951,-88.5695,Strong to severe thunderstorms moved across pa...,A tornado touched down in the extreme northern...,CSV,2010,StormEvents_details-ftp_v1.0_d2010_c20260323.c...
1,201007,7,1251,201007,7,1630,43850,254780,NEW HAMPSHIRE,33,...,None,NaN,NaN,NaN,NaN,A strong ridge built into Southern New England...,Heat index values at the Nashua Boire Field (K...,CSV,2010,StormEvents_details-ftp_v1.0_d2010_c20260323.c...
2,201001,17,2300,201001,18,1500,36500,211550,NEW HAMPSHIRE,33,...,None,NaN,NaN,NaN,NaN,A coastal storm passing southern New England j...,Four to eight inches fell across eastern Hills...,CSV,2010,StormEvents_details-ftp_v1.0_d2010_c20260323.c...
3,201010,1,830,201010,1,1000,44854,260014,NEW HAMPSHIRE,33,...,None,NaN,NaN,NaN,NaN,Several waves of low pressure moved across Sou...,"In Manchester, firefighters responded to about...",CSV,2010,StormEvents_details-ftp_v1.0_d2010_c20260323.c...
4,201007,6,951,201007,6,1830,43850,254779,NEW HAMPSHIRE,33,...,None,NaN,NaN,NaN,NaN,A strong ridge built into Southern New England...,Heat index values at the Manchester Airport (K...,CSV,2010,StormEvents_details-ftp_v1.0_d2010_c20260323.c...


Compruebo que estén disponibles todas las columnas necesarias


In [7]:
required_columns = [
    "event_id", "episode_id", "source_year", "begin_yearmonth", "begin_day", "begin_time", "damage_property", "damage_crops", "state",
    "event_type", "cz_type", "magnitude", "magnitude_type", "begin_lat", "begin_lon",
    "begin_range", "begin_azimuth", "wfo", "cz_timezone", "flood_cause"]

missing_columns = []

for column in required_columns:
    if column not in df_storm.columns:
        missing_columns.append(column)

print("Columnas faltantes:", missing_columns)

Columnas faltantes: []


In [8]:
if len(missing_columns) > 0:
    raise ValueError(f"Faltan columnas necesarias: {missing_columns}")

Reviso los formatos utilizados para registrar los daños


In [9]:
def classify_damage_format(value):
    if pd.isna(value):
        return "MISSING"
    text = str(value)
    text = text.strip()
    text = text.upper()

    if text.endswith("K"):
        return "K"

    if text.endswith("M"):
        return "M"

    if text.endswith("B"):
        return "B"

    try:
        float(text)
        return "NO_SUFFIX"

    except ValueError:
        return "OTHER"

In [10]:
property_format = (df_storm["damage_property"].map(classify_damage_format).value_counts())

crop_format = (df_storm["damage_crops"].map(classify_damage_format).value_counts())

print("Daños materiales")
display(property_format)

print("Daños agrícolas")
display(crop_format)

Daños materiales


damage_property
K            826781
MISSING      204143
M              6578
NO_SUFFIX       131
B                58
Name: count, dtype: int64

Daños agrícolas


damage_crops
K            828591
MISSING      206757
M              2209
NO_SUFFIX       130
B                 4
Name: count, dtype: int64

Defino una función para convertir los daños a valores numéricos


In [11]:
def convert_damage(value):
    if pd.isna(value):
        return None

    text = str(value)
    text = text.strip()
    text = text.upper()
    text = text.replace(",", "")
    text = text.replace("$", "")

    if text == "":
        return None

    multiplier = 1
    suffix = text[-1]

    if suffix == "K":
        multiplier = 1_000
        text = text[:-1]

    elif suffix == "M":
        multiplier = 1_000_000
        text = text[:-1]

    elif suffix == "B":
        multiplier = 1_000_000_000
        text = text[:-1]

    try:
        number = float(text)
        return number * multiplier

    except ValueError:
        return None

Compruebo la conversión con distintos formatos de daños


In [12]:
print(convert_damage("10.00K"))
print(convert_damage("2.50M"))
print(convert_damage("500000"))
print(convert_damage("4500000"))
print(convert_damage(None))

10000.0
2500000.0
500000.0
4500000.0
None


Convierto los daños de propiedades y cultivos a valores numéricos


In [13]:
df_storm["damage_property_usd"] = (df_storm["damage_property"].map(convert_damage))

df_storm["damage_crops_usd"] = (df_storm["damage_crops"].map(convert_damage))

Compruebo si quedaron valores de daños sin convertir


In [14]:
invalid_property = (df_storm["damage_property"].notna() & df_storm["damage_property_usd"].isna())

invalid_crops = (df_storm["damage_crops"].notna() & df_storm["damage_crops_usd"].isna())

print("Daños materiales no convertidos:", invalid_property.sum())

print("Daños agrícolas no convertidos:", invalid_crops.sum())

Daños materiales no convertidos: 0
Daños agrícolas no convertidos: 0


In [15]:
total_invalid = (invalid_property.sum() + invalid_crops.sum())

if total_invalid > 0:
    raise ValueError("Hay valores de daños que no pudieron convertirse.")

Creo un indicador para los eventos sin valores de daños informados


In [16]:
df_storm["damage_values_missing"] = (df_storm["damage_property"].isna() & df_storm["damage_crops"].isna()).astype("int8")

Reemplazo por cero los daños faltantes después de identificarlos


In [17]:
df_storm["damage_property_usd"] = (df_storm["damage_property_usd"].fillna(0))

df_storm["damage_crops_usd"] = (df_storm["damage_crops_usd"].fillna(0))

Calculo el daño total y creo el target binario


In [18]:
df_storm["total_damage_usd"] = (df_storm["damage_property_usd"] + df_storm["damage_crops_usd"])

df_storm["has_recorded_damage"] = (df_storm["total_damage_usd"] > 0).astype("int8")

Reviso una muestra de los daños convertidos y el target


In [20]:
df_storm[["damage_property", "damage_property_usd", "damage_crops", "damage_crops_usd", "total_damage_usd", "has_recorded_damage"]].head(20)

,damage_property,damage_property_usd,damage_crops,damage_crops_usd,total_damage_usd,has_recorded_damage
0,0,0.0,0,0.0,0.0,0
1,0.00K,0.0,0.00K,0.0,0.0,0
2,0.00K,0.0,0.00K,0.0,0.0,0
3,50.00K,50000.0,0.00K,0.0,50000.0,1
4,0.00K,0.0,0.00K,0.0,0.0,0
5,0.00K,0.0,0.00K,0.0,0.0,0
6,2.50M,2500000.0,0.00K,0.0,2500000.0,1
7,0.00K,0.0,0.00K,0.0,0.0,0
8,10.00K,10000.0,0.00K,0.0,10000.0,1
9,500000,500000.0,0,0.0,500000.0,1


Compruebo la distribución del target


In [21]:
target_report = (df_storm["has_recorded_damage"].value_counts().sort_index().reset_index())

target_report.columns = ["has_recorded_damage", "rows"]

target_report["percentage"] = (target_report["rows"]/ target_report["rows"].sum()* 100).round(2)

target_report

,has_recorded_damage,rows,percentage
0,0,808152,77.88
1,1,229539,22.12


Compruebo la proporción de eventos con daños por año


In [22]:
damage_by_year = (df_storm.groupby("source_year")["has_recorded_damage"].agg(["count", "sum", "mean"]).reset_index())

damage_by_year["mean"] = (damage_by_year["mean"] * 100).round(2)

damage_by_year.columns = ["year", "events", "events_with_damage", "damage_percentage" ]

damage_by_year

,year,events,events_with_damage,damage_percentage
0,2010,62809,17251,27.47
1,2011,79091,22140,27.99
2,2012,64503,16048,24.88
3,2013,59986,14982,24.98
4,2014,59475,13535,22.76
5,2015,57907,13076,22.58
6,2016,56005,12021,21.46
7,2017,57041,13598,23.84
8,2018,62699,12014,19.16
9,2019,67864,14287,21.05


Convierto a numéricas las variables necesarias para reconstruir la fecha


In [23]:
time_columns = ["begin_yearmonth", "begin_day", "begin_time"]

for column in time_columns:
    df_storm[column] = pd.to_numeric(df_storm[column], errors="coerce")

Creo variables de año, mes, día de la semana y hora


In [24]:
df_storm["event_year"] = (df_storm["begin_yearmonth"] // 100)

df_storm["begin_month"] = (df_storm["begin_yearmonth"] % 100)

df_storm["begin_hour"] = (df_storm["begin_time"] // 100)

df_storm["begin_minute"] = (df_storm["begin_time"] % 100)

Reconstruyo la fecha y hora de inicio de cada evento


In [25]:
date_parts = pd.DataFrame({
    "year": df_storm["event_year"],
    "month": df_storm["begin_month"],
    "day": df_storm["begin_day"],
    "hour": df_storm["begin_hour"],
    "minute": df_storm["begin_minute"]})

df_storm["begin_datetime"] = pd.to_datetime(date_parts, errors="coerce")

df_storm["begin_day_of_week"] = (df_storm["begin_datetime"].dt.dayofweek)

Compruebo los valores faltantes de las variables temporales


In [26]:
temporal_report = pd.DataFrame({
    "missing_rows": 
        df_storm[["event_year", "begin_month", "begin_day_of_week", "begin_hour", "begin_datetime"]].isna().sum(),
    "missing_percentage": (
        df_storm[["event_year", "begin_month", "begin_day_of_week", "begin_hour", "begin_datetime"]].isna().mean() * 100).round(2)})

temporal_report

,missing_rows,missing_percentage
event_year,0,0.0
begin_month,0,0.0
begin_day_of_week,0,0.0
begin_hour,0,0.0
begin_datetime,0,0.0


In [27]:
df_storm[["begin_yearmonth", "begin_day", "begin_time", "begin_datetime", "event_year", "begin_month", "begin_day_of_week", "begin_hour"]].head(20)

,begin_yearmonth,begin_day,begin_time,begin_datetime,event_year,begin_month,begin_day_of_week,begin_hour
0,201011,22,1531,2010-11-22 15:31:00,2010,11,0,15
1,201007,7,1251,2010-07-07 12:51:00,2010,7,2,12
2,201001,17,2300,2010-01-17 23:00:00,2010,1,6,23
3,201010,1,830,2010-10-01 08:30:00,2010,10,4,8
4,201007,6,951,2010-07-06 09:51:00,2010,7,1,9
5,201012,26,1700,2010-12-26 17:00:00,2010,12,6,17
6,201002,25,2305,2010-02-25 23:05:00,2010,2,3,23
7,201002,16,1200,2010-02-16 12:00:00,2010,2,1,12
8,201003,14,1345,2010-03-14 13:45:00,2010,3,6,13
9,201011,22,1500,2010-11-22 15:00:00,2010,11,0,15


Convierto las variables numéricas al formato adecuado


In [28]:
numeric_source_columns = ["magnitude", "begin_lat", "begin_lon", "begin_range"]

for column in numeric_source_columns:
    df_storm[column] = pd.to_numeric(df_storm[column], errors="coerce")

Creo indicadores para los valores faltantes de magnitud, coordenadas y distancia


In [29]:
df_storm["magnitude_missing"] = (df_storm["magnitude"].isna()).astype("int8")

df_storm["coordinates_missing"] = (df_storm["begin_lat"].isna() | df_storm["begin_lon"].isna()).astype("int8")

df_storm["begin_range_missing"] = (df_storm["begin_range"].isna()).astype("int8")

Normalizo las variables categóricas


In [30]:
categorical_features = ["month_name", "state", "event_type", "cz_type", "magnitude_type", "wfo", "cz_timezone", "flood_cause", "begin_azimuth"]

for column in categorical_features:
    text_values = (df_storm[column].astype("string").str.strip().str.upper())
    text_values = text_values.replace("", pd.NA)
    df_storm[column] = text_values.fillna("MISSING")

Defino las variables numéricas, categóricas, identificadores y target del modelo


In [31]:
numeric_features = ["event_year", "begin_day_of_week", "begin_hour", "magnitude", "begin_lat", "begin_lon", "begin_range", "magnitude_missing", "coordinates_missing", "begin_range_missing"]

target = "has_recorded_damage"

identifier_columns = ["event_id", "episode_id", "source_year"]

Compruebo la cantidad de variables seleccionadas


In [32]:
print("Variables numéricas:", len(numeric_features))

print("Variables categóricas:", len(categorical_features))

print("Variables predictoras totales:", len(numeric_features) + len(categorical_features))

Variables numéricas: 10
Variables categóricas: 9
Variables predictoras totales: 19


Creo la base reducida que utilizaré para modelado


In [33]:
model_columns = (identifier_columns + numeric_features + categorical_features + [target])

df_model_base = df_storm[model_columns].copy()

print("Filas:", f"{df_model_base.shape[0]:,}")
print("Columnas:", df_model_base.shape[1])

Filas: 1,037,691
Columnas: 23


In [34]:
df_model_base.head()

,event_id,episode_id,source_year,event_year,begin_day_of_week,begin_hour,magnitude,begin_lat,begin_lon,begin_range,...,month_name,state,event_type,cz_type,magnitude_type,wfo,cz_timezone,flood_cause,begin_azimuth,has_recorded_damage
0,268262,46247,2010,2010,0,15,NaN,42.4932,-88.5704,1.0,...,NOVEMBER,ILLINOIS,TORNADO,C,MISSING,LOT,CST-6,MISSING,SE,0
1,254780,43850,2010,2010,2,12,NaN,NaN,NaN,NaN,...,JULY,NEW HAMPSHIRE,HEAT,Z,MISSING,BOX,EST-5,MISSING,MISSING,0
2,211550,36500,2010,2010,6,23,NaN,NaN,NaN,NaN,...,JANUARY,NEW HAMPSHIRE,HEAVY SNOW,Z,MISSING,BOX,EST-5,MISSING,MISSING,0
3,260014,44854,2010,2010,4,8,45.0,NaN,NaN,NaN,...,OCTOBER,NEW HAMPSHIRE,STRONG WIND,Z,EG,BOX,EST-5,MISSING,MISSING,1
4,254779,43850,2010,2010,1,9,NaN,NaN,NaN,NaN,...,JULY,NEW HAMPSHIRE,HEAT,Z,MISSING,BOX,EST-5,MISSING,MISSING,0


Compruebo los valores faltantes de la base de modelado


In [35]:
missing_model = pd.DataFrame({
    "missing_rows": 
        df_model_base.isna().sum(),
    "missing_percentage": (
        df_model_base.isna().mean() * 100).round(2)})

missing_model = missing_model.sort_values("missing_percentage",ascending=False)

missing_model

,missing_rows,missing_percentage
magnitude,494484,47.65
begin_lat,401645,38.71
begin_lon,401645,38.71
begin_range,401617,38.70
event_id,0,0.00
state,0,0.00
begin_azimuth,0,0.00
flood_cause,0,0.00
cz_timezone,0,0.00
wfo,0,0.00


Reviso la cantidad de categorías de cada variable categórica


In [36]:
for column in categorical_features:
    categories = df_model_base[column].nunique()
    print(f"{column}: {categories} categorías")

month_name: 12 categorías
state: 69 categorías
event_type: 54 categorías
cz_type: 2 categorías
magnitude_type: 5 categorías
wfo: 123 categorías
cz_timezone: 12 categorías
flood_cause: 8 categorías
begin_azimuth: 17 categorías


Registro las columnas que excluyo del modelo


In [37]:
excluded_from_model = [
    "damage_property", "damage_crops", "damage_property_usd", "damage_crops_usd", "total_damage_usd", "damage_values_missing", "injuries_direct",
    "injuries_indirect", "deaths_direct", "deaths_indirect", "tor_f_scale", "episode_narrative", "event_narrative",
    "begin_date_time", "end_date_time", "end_lat", "end_lon", "event_id", "episode_id"]

excluded_from_model

['damage_property',
 'damage_crops',
 'damage_property_usd',
 'damage_crops_usd',
 'total_damage_usd',
 'damage_values_missing',
 'injuries_direct',
 'injuries_indirect',
 'deaths_direct',
 'deaths_indirect',
 'tor_f_scale',
 'episode_narrative',
 'event_narrative',
 'begin_date_time',
 'end_date_time',
 'end_lat',
 'end_lon',
 'event_id',
 'episode_id']

Defino los archivos de salida


In [38]:
PREPARED_FILE = (PROCESSED_DIR / "storm_events_prepared_2010_2025.parquet")

MODEL_FILE = (PROCESSED_DIR / "storm_events_model_base_2010_2025.parquet")

Guardo la base preparada y la base reducida para modelado


In [40]:
df_storm.to_parquet(PREPARED_FILE,index=False)

df_model_base.to_parquet(MODEL_FILE,index=False)

print("Guardada Base completa")

print("Guardada Base de modelo")

Guardada Base completa
Guardada Base de modelo


Compruebo que la base de modelado se haya guardado correctamente


In [41]:
df_model_check = pd.read_parquet(MODEL_FILE)

assert (df_model_check.shape == df_model_base.shape)

print("Archivo de modelado validado:", df_model_check.shape)

del df_model_check

Archivo de modelado validado: (1037691, 23)
